# Cardiac Ultrasound — Full Inference Pipeline

This notebook runs the **trained models** on new ultrasound images.

**You do NOT need the training dataset (database_nifti/) to use this notebook.**

### What you need (get all `.pt` files from your teammate):
| Weight File | Purpose |
|---|---|
| `segmentation_weights.pt` | U-Net segmentation (LV, myocardium, LA) |
| `cactus_resnet18_classification.pt` | Classify ultrasound view type (2CH, 4CH, etc.) |
| `cactus_resnet18_regression.pt` | Predict image quality score for each angle |
| `ef_trained.pt` | Predict Ejection Fraction directly |
| `multitask_ultrasound_model.pt` | Combined multi-task model |

- Your ultrasound images (`.nii.gz`, `.png`, `.jpg`, or `.dcm`)

### What this does:
1. **Classifies** what ultrasound view/angle the image shows
2. **Assesses quality** — is this a good angle?
3. **Segments** cardiac structures (LV cavity, myocardium, left atrium)
4. **Calculates metrics** (areas, volumes, EF, VTI, cardiac output)
5. **Visualizes** results with color overlays

In [6]:
# Cell [1] — Create a local virtual environment and install packages
import subprocess, sys, os

venv_path = os.path.expanduser("~/cardiac_venv")

# Create venv (only needs to run once)
if not os.path.exists(venv_path):
    subprocess.run([sys.executable, "-m", "venv", venv_path], check=True)
    print(f"✅ Created venv at {venv_path}")

# Install packages into the venv
pip_path = os.path.join(venv_path, "bin", "pip")
subprocess.run([pip_path, "install", "torch", "torchvision", "numpy", "nibabel", "matplotlib", "Pillow", "opencv-python-headless"], check=True)

# Add venv packages to this notebook's path
import site
site_packages = os.path.join(venv_path, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
site.addsitedir(site_packages)

print(f"\n✅ All packages installed and available!")
print(f"   Location: {site_packages}")

✅ Created venv at /home/users/leah26/cardiac_venv
  Using cached nibabel-5.3.3-py3-none-any.whl.metadata (9.1 kB)
  Using cached filelock-3.24.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.m


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /home/users/leah26/cardiac_venv/bin/python3.13 -m pip install --upgrade pip



✅ All packages installed and available!
   Location: /home/users/leah26/cardiac_venv/lib/python3.13/site-packages


In [7]:
# Cell [2] — Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision.transforms import functional as F
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cpu


In [ ]:
# Cell [3] — Configuration
# =====================================================
# UPDATE THESE PATHS to where YOU saved each .pt file
# =====================================================

WEIGHTS_DIR = "weights/"  # Folder containing all .pt files — CHANGE THIS

SEG_MODEL_PATH   = os.path.join(WEIGHTS_DIR, "segmentation_weights.pt")
CLASS_MODEL_PATH  = os.path.join(WEIGHTS_DIR, "cactus_resnet18_classification.pt")
REGR_MODEL_PATH   = os.path.join(WEIGHTS_DIR, "cactus_resnet18_regression.pt")
EF_MODEL_PATH     = os.path.join(WEIGHTS_DIR, "ef_trained.pt")
MULTI_MODEL_PATH  = os.path.join(WEIGHTS_DIR, "multitask_ultrasound_model.pt")

IMAGE_DIR = "test_images/"  # Folder with your ultrasound images — CHANGE THIS

IMG_SIZE = (256, 256)

LABEL_MAP = {
    0: 'Background',
    1: 'LV Cavity',
    2: 'Myocardium',
    3: 'Left Atrium'
}

print("Config loaded")
print(f"  Weights dir:  {WEIGHTS_DIR}")
print(f"  Image dir:    {IMAGE_DIR}")
print(f"  Input size:   {IMG_SIZE}")
print()
for name, path in [("Segmentation", SEG_MODEL_PATH),
                    ("Classification", CLASS_MODEL_PATH),
                    ("Regression", REGR_MODEL_PATH),
                    ("EF", EF_MODEL_PATH),
                    ("Multitask", MULTI_MODEL_PATH)]:
    found = os.path.exists(path)
    status = "FOUND" if found else "NOT FOUND"
    print(f"  {name:<16} {status:<12} {path}")

In [ ]:
# Cell [3b] — Inspect weight files to discover architectures

def inspect_weights(path, label=""):
    """Print summary of a .pt weight file's contents."""
    if not os.path.exists(path):
        print(f"  [{label}] File not found: {path}\n")
        return None

    try:
        data = torch.load(path, map_location="cpu", weights_only=False)
    except RuntimeError as e:
        print(f"  [{label}] {path}")
        print(f"    FAILED to load: {e}")
        print(f"    (File may be corrupted or saved in an unsupported format.)")
        print(f"    Try re-downloading this file from your teammate.\n")
        return None

    print(f"  [{label}] {path}")
    print(f"    Top-level type: {type(data).__name__}")

    state_dict = None
    if isinstance(data, dict):
        top_keys = list(data.keys())[:20]
        print(f"    Keys: {top_keys}")
        if "state_dict" in data:
            state_dict = data["state_dict"]
        elif "model_state_dict" in data:
            state_dict = data["model_state_dict"]
        elif all(isinstance(v, torch.Tensor) for v in list(data.values())[:5]):
            state_dict = data
    elif isinstance(data, nn.Module):
        print(f"    Saved as full model: {type(data).__name__}")
        state_dict = data.state_dict()

    if state_dict is not None:
        keys = list(state_dict.keys())
        print(f"    Total layers: {len(keys)}")
        print(f"    All Conv2d weight shapes:")
        for k in keys:
            if "weight" in k and state_dict[k].ndim == 4:
                print(f"      {k}: {list(state_dict[k].shape)}")
        print(f"    Final layer:")
        for k in keys[-3:]:
            print(f"      {k}: {list(state_dict[k].shape) if state_dict[k].ndim > 0 else 'scalar'}")
    print()
    return data

print("=== Weight File Inspection ===\n")
for label, path in [("Segmentation", SEG_MODEL_PATH),
                     ("Classification", CLASS_MODEL_PATH),
                     ("Regression", REGR_MODEL_PATH),
                     ("EF", EF_MODEL_PATH),
                     ("Multitask", MULTI_MODEL_PATH)]:
    inspect_weights(path, label)

In [ ]:
# Cell [4] — Model Architectures
# Matched to the ACTUAL weight files discovered by Cell [3b].

import torchvision.models as models

# ---------------------------------------------------------------------------
# Segmentation: SimpleUNet  (matches segmentation_weights.pt)
#   Keys: down1, down2, mid, up2, up1, out
#   No BatchNorm, 2-level encoder, bilinear upsampling, 1-channel output
# ---------------------------------------------------------------------------

class SimpleUNet(nn.Module):
    """
    Lightweight U-Net that matches segmentation_weights.pt.
    Auto-detects channel sizes from a state dict if provided.
    """
    def __init__(self, in_ch=1, out_ch=1, features=(64, 128, 256)):
        super().__init__()
        f1, f2, f3 = features
        self.down1 = self._block(in_ch, f1)
        self.down2 = self._block(f1, f2)
        self.mid   = self._block(f2, f3)
        self.up2   = self._block(f3 + f2, f2)
        self.up1   = self._block(f2 + f1, f1)
        self.out   = nn.Conv2d(f1, out_ch, 1)
        self.pool  = nn.MaxPool2d(2)

    @staticmethod
    def _block(in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        m  = self.mid(self.pool(d2))
        u2 = self.up2(torch.cat([
            nn.functional.interpolate(m, size=d2.shape[2:], mode='bilinear', align_corners=True),
            d2], dim=1))
        u1 = self.up1(torch.cat([
            nn.functional.interpolate(u2, size=d1.shape[2:], mode='bilinear', align_corners=True),
            d1], dim=1))
        return self.out(u1)

    @classmethod
    def from_state_dict(cls, sd):
        """Build a SimpleUNet whose channels match the given state dict."""
        in_ch = sd['down1.0.weight'].shape[1]
        f1    = sd['down1.0.weight'].shape[0]
        f2    = sd['down2.0.weight'].shape[0]
        f3    = sd['mid.0.weight'].shape[0]
        out_ch = sd['out.weight'].shape[0]
        model = cls(in_ch, out_ch, (f1, f2, f3))
        model.load_state_dict(sd)
        return model


# ---------------------------------------------------------------------------
# Classification / Regression: ResNet18-based
# ---------------------------------------------------------------------------

def _extract_state_dict(raw):
    """Pull the actual state dict out of whatever torch.load returned."""
    if isinstance(raw, dict):
        if "state_dict" in raw:
            return raw["state_dict"]
        if "model_state_dict" in raw:
            return raw["model_state_dict"]
        if all(isinstance(v, torch.Tensor) for v in list(raw.values())[:3]):
            return raw
    return None


def load_resnet18_from_weights(path, fallback_outputs=1):
    """
    Load a ResNet18 from a .pt file, auto-detecting:
      - number of input channels (1 for grayscale, 3 for RGB)
      - number of output classes/values
    """
    if not os.path.exists(path):
        print(f"  File not found: {path}")
        return None

    try:
        raw = torch.load(path, map_location=DEVICE, weights_only=False)
    except RuntimeError as e:
        print(f"  Failed to load {path}: {e}")
        return None

    sd = _extract_state_dict(raw)
    if sd is None:
        print(f"  Unexpected format in {path} (type={type(raw).__name__})")
        return None

    # Auto-detect shapes
    in_ch = sd['conv1.weight'].shape[1] if 'conv1.weight' in sd else 1
    n_out = sd['fc.weight'].shape[0] if 'fc.weight' in sd else fallback_outputs

    net = models.resnet18(weights=None)
    net.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)
    net.fc = nn.Linear(net.fc.in_features, n_out)
    net.load_state_dict(sd, strict=False)
    net = net.to(DEVICE)
    net.eval()
    return net, n_out, in_ch


print(f"SimpleUNet defined  (matches segmentation_weights.pt)")
print(f"ResNet18 loader defined (auto-detects input channels & output size)")

In [ ]:
# Cell [5] — Load All Trained Models

loaded_models = {}
model_info = {}

# 1. Segmentation (SimpleUNet — 1-channel binary output)
if os.path.exists(SEG_MODEL_PATH):
    try:
        seg_sd = torch.load(SEG_MODEL_PATH, map_location=DEVICE, weights_only=False)
        sd = _extract_state_dict(seg_sd) if not all(
            isinstance(v, torch.Tensor) for v in list(seg_sd.values())[:3]
        ) else seg_sd
        if sd is None:
            sd = seg_sd
        seg_model = SimpleUNet.from_state_dict(sd).to(DEVICE)
        seg_model.eval()
        out_ch = sd['out.weight'].shape[0]
        loaded_models["segmentation"] = seg_model
        model_info["segmentation"] = {"out_channels": out_ch}
        print(f"Segmentation model loaded ({out_ch}-channel output) from {SEG_MODEL_PATH}")
    except Exception as e:
        print(f"Segmentation model failed to load: {e}")
else:
    print(f"Segmentation weights not found at {SEG_MODEL_PATH}")

# 2. Classification (ResNet18 — 3-class output)
if os.path.exists(CLASS_MODEL_PATH):
    result = load_resnet18_from_weights(CLASS_MODEL_PATH)
    if result:
        class_model, n_classes, in_ch = result
        loaded_models["classification"] = class_model
        model_info["classification"] = {"n_classes": n_classes, "in_channels": in_ch}
        print(f"Classification model loaded ({n_classes} classes, {in_ch}-ch input) from {CLASS_MODEL_PATH}")
else:
    print(f"Classification weights not found at {CLASS_MODEL_PATH}")

# 3. Regression (ResNet18 — quality score)
if os.path.exists(REGR_MODEL_PATH):
    result = load_resnet18_from_weights(REGR_MODEL_PATH)
    if result:
        regr_model, n_out, in_ch = result
        loaded_models["regression"] = regr_model
        model_info["regression"] = {"n_outputs": n_out, "in_channels": in_ch}
        print(f"Regression model loaded ({n_out} outputs, {in_ch}-ch input) from {REGR_MODEL_PATH}")
    else:
        print(f"Regression file could not be loaded — it may be corrupted.")
        print(f"  Ask your teammate to re-share this file.")
else:
    print(f"Regression weights not found at {REGR_MODEL_PATH}")

# 4. EF model
if os.path.exists(EF_MODEL_PATH):
    result = load_resnet18_from_weights(EF_MODEL_PATH)
    if result:
        ef_model, n_out, in_ch = result
        loaded_models["ef"] = ef_model
        model_info["ef"] = {"n_outputs": n_out, "in_channels": in_ch}
        print(f"EF model loaded ({n_out} outputs, {in_ch}-ch input) from {EF_MODEL_PATH}")
    else:
        print(f"EF weights format not recognized — check Cell [3b] output")
else:
    print(f"EF weights not found at {EF_MODEL_PATH}")

# 5. Multitask model (architecture unknown — inspect first)
if os.path.exists(MULTI_MODEL_PATH):
    print(f"Multitask weights found at {MULTI_MODEL_PATH}")
    print("  Check Cell [3b] output to determine the architecture, then add loading code here.")
else:
    print(f"Multitask weights not found at {MULTI_MODEL_PATH}")

print(f"\nLoaded {len(loaded_models)} model(s): {list(loaded_models.keys())}")
print(f"Model info: {model_info}")

In [ ]:
# Cell [6] — Image Loading Helpers
# Supports images AND video clips from the ultrasound probe

def load_ultrasound_image(path, frame_index=None):
    """
    Load an ultrasound image from various formats, including MP4 video.

    For video files (.mp4, .avi, .mov):
      - Extracts a single frame (middle frame by default).
      - Set frame_index to pick a specific frame (0-based).

    Returns: (image_array, pixel_spacing)
        image_array: 2D numpy array (H, W), grayscale
        pixel_spacing: (dx, dy) in mm, or (1.0, 1.0) if unknown
    """
    path = str(path)
    ext = path.lower()

    if ext.endswith(('.mp4', '.avi', '.mov', '.mkv')):
        import cv2
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            raise IOError(f"Cannot open video: {path}")

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        target = frame_index if frame_index is not None else total_frames // 2
        target = max(0, min(target, total_frames - 1))

        cap.set(cv2.CAP_PROP_POS_FRAMES, target)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            raise IOError(f"Failed to read frame {target} from {path}")

        img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)
        return img, (1.0, 1.0)

    elif ext.endswith(('.nii', '.nii.gz')):
        import nibabel as nib
        nii = nib.load(path)
        img = nii.get_fdata().astype(np.float32)
        if img.ndim == 3:
            img = img[:, :, img.shape[2] // 2]
        spacing = nii.header.get_zooms()[:2]
        return img, spacing

    elif ext.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
        from PIL import Image
        img = np.array(Image.open(path).convert('L')).astype(np.float32)
        return img, (1.0, 1.0)

    elif ext.endswith('.dcm'):
        try:
            import pydicom
            ds = pydicom.dcmread(path)
            img = ds.pixel_array.astype(np.float32)
            spacing = getattr(ds, 'PixelSpacing', [1.0, 1.0])
            return img, (float(spacing[0]), float(spacing[1]))
        except ImportError:
            raise ImportError("Install pydicom: pip install pydicom")

    elif ext.endswith('.npy'):
        img = np.load(path).astype(np.float32)
        return img, (1.0, 1.0)

    else:
        raise ValueError(f"Unsupported format: {path}")


def get_video_info(path):
    """Print basic info about a video file."""
    import cv2
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        print(f"  Cannot open: {path}")
        return
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    duration = frames / fps if fps > 0 else 0
    print(f"  {Path(path).name}: {w}x{h}, {frames} frames, {fps:.1f} fps, {duration:.1f}s")


print("Image/video loader ready")
print("  Supported: .mp4, .avi, .mov, .nii.gz, .nii, .png, .jpg, .dcm, .npy")
print("  Videos: extracts middle frame by default (set frame_index= to pick a specific frame)")

In [ ]:
# Cell [7] — Segmentation Inference
# Handles BOTH 1-channel (binary) and multi-channel (multi-class) output

def predict_segmentation(seg_model, image_array):
    """
    Run segmentation on a single 2D ultrasound image.

    Automatically handles:
      - 1-channel output (binary, sigmoid + threshold)
      - N-channel output (multi-class, softmax + argmax)

    Returns:
        pred_mask: 2D int array (H, W) with labels 0/1 (binary) or 0..N-1
        confidence: 2D float array (H, W) with prediction confidence
    """
    original_shape = image_array.shape[:2]

    img = image_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
    img_tensor = F.resize(img_tensor, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR)
    img_tensor = img_tensor.to(DEVICE)

    with torch.no_grad():
        output = seg_model(img_tensor)  # (1, C, H, W)

    n_classes = output.shape[1]

    if n_classes == 1:
        prob = torch.sigmoid(output)                         # (1, 1, H, W)
        pred = (prob > 0.5).float()
        confidence = torch.where(pred == 1, prob, 1.0 - prob)
        pred = pred.squeeze(1)           # (1, H, W)
        confidence = confidence.squeeze(1)
    else:
        probs = torch.softmax(output, dim=1)
        confidence, pred = torch.max(probs, dim=1)           # (1, H, W)

    pred = F.resize(pred.unsqueeze(1).float(), original_shape,
                    interpolation=F.InterpolationMode.NEAREST)
    confidence = F.resize(confidence.unsqueeze(1).float(), original_shape,
                          interpolation=F.InterpolationMode.BILINEAR)

    return (
        pred.squeeze().cpu().numpy().astype(np.int64),
        confidence.squeeze().cpu().numpy(),
    )

seg_out_ch = model_info.get("segmentation", {}).get("out_channels", "?")
print(f"Segmentation inference ready ({seg_out_ch}-channel output -> "
      f"{'binary (LV mask)' if seg_out_ch == 1 else 'multi-class'})")

In [ ]:
# Cell [7b] — Classification, Regression & EF Inference

def _prepare_resnet_input(image_array, model_key):
    """Normalize and shape an image for a ResNet model, handling 1-ch vs 3-ch input."""
    img = image_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    in_ch = model_info.get(model_key, {}).get("in_channels", 1)
    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
    if in_ch == 3:
        img_tensor = img_tensor.expand(-1, 3, -1, -1)            # (1, 3, H, W)

    img_tensor = F.resize(img_tensor, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR)
    return img_tensor.to(DEVICE)


def predict_view_class(image_array):
    """
    Classify ultrasound view type.
    Returns (predicted_class_index, class_probabilities).
    """
    if "classification" not in loaded_models:
        return None, None

    img_tensor = _prepare_resnet_input(image_array, "classification")
    with torch.no_grad():
        logits = loaded_models["classification"](img_tensor)
        probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
    return int(probs.argmax()), probs


def predict_quality_score(image_array):
    """
    Predict image quality score.
    Returns a float (interpretation depends on training).
    """
    if "regression" not in loaded_models:
        return None

    img_tensor = _prepare_resnet_input(image_array, "regression")
    with torch.no_grad():
        score = loaded_models["regression"](img_tensor)
    return float(score.squeeze().cpu())


def predict_ef_direct(image_array):
    """
    Predict Ejection Fraction directly.
    Returns a float EF value.
    """
    if "ef" not in loaded_models:
        return None

    img_tensor = _prepare_resnet_input(image_array, "ef")
    with torch.no_grad():
        ef_val = loaded_models["ef"](img_tensor)
    return float(ef_val.squeeze().cpu())


available = []
if "classification" in loaded_models:
    n = model_info["classification"]["n_classes"]
    available.append(f"View classification ({n} classes)")
if "regression" in loaded_models:
    available.append("Quality regression")
if "ef" in loaded_models:
    available.append("Direct EF prediction")
print(f"Inference functions ready: {available or ['Only segmentation (other models not loaded)']}")

In [ ]:
# Cell [8] — Cardiac Metrics Calculation
# Works for both binary masks (label 1 = LV) and multi-class masks (1=LV, 2=Myo, 3=LA)

def calculate_areas(pred_mask, pixel_spacing_mm=(1.0, 1.0)):
    """Calculate areas of each segmented structure in mm2."""
    dx, dy = pixel_spacing_mm
    pixel_area = dx * dy

    areas = {'lv_area_mm2': float((pred_mask == 1).sum() * pixel_area)}
    if pred_mask.max() > 1:
        areas['myocardium_area_mm2'] = float((pred_mask == 2).sum() * pixel_area)
        areas['la_area_mm2'] = float((pred_mask == 3).sum() * pixel_area)
    return areas


def calculate_lv_length_mm(pred_mask, pixel_spacing_mm=(1.0, 1.0)):
    """Estimate LV long-axis length from segmentation mask."""
    dx, dy = pixel_spacing_mm
    ys, xs = np.where(pred_mask == 1)

    if len(xs) == 0:
        return None

    length_pixels = np.sqrt((xs.max() - xs.min())**2 + (ys.max() - ys.min())**2)
    return float(length_pixels * np.mean([dx, dy]))


def calculate_volume_biplane(area_2ch_mm2, area_4ch_mm2, length_mm):
    """
    Biplane area-length method (modified Simpson's):
    V = (8 / 3π) × (A_2CH × A_4CH) / L
    Returns volume in mL.
    """
    if None in [area_2ch_mm2, area_4ch_mm2, length_mm] or length_mm == 0:
        return None
    
    volume_mm3 = (8 / (3 * np.pi)) * (area_2ch_mm2 * area_4ch_mm2) / length_mm
    return volume_mm3 / 1000  # mm³ → mL


def calculate_ef(edv_ml, esv_ml):
    """Ejection Fraction (%)."""
    if edv_ml is None or esv_ml is None or edv_ml == 0:
        return None
    return ((edv_ml - esv_ml) / edv_ml) * 100


def calculate_cardiac_output(sv_ml, heart_rate_bpm):
    """Cardiac Output in L/min."""
    return (sv_ml * heart_rate_bpm) / 1000


def calculate_vti(sv_ml, lvot_diameter_cm=2.0):
    """
    Velocity Time Integral (cm).
    VTI = SV / LVOT_area
    """
    lvot_area = np.pi * (lvot_diameter_cm / 2)**2
    return sv_ml / lvot_area  # mL = cm³, so result is in cm


print("✅ Cardiac metrics functions ready")

In [ ]:
# Cell [9] — Visualization
# Handles both binary (1-class) and multi-class segmentation

def visualize_result(image, pred_mask, confidence, title=""):
    """Display original image, segmentation, and overlay."""
    is_binary = pred_mask.max() <= 1

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    if is_binary:
        axes[1].imshow(pred_mask, cmap='Reds', vmin=0, vmax=1)
        axes[1].set_title('Segmentation (LV)')
    else:
        axes[1].imshow(pred_mask, cmap='tab10', vmin=0, vmax=pred_mask.max())
        axes[1].set_title('Segmentation')
    axes[1].axis('off')

    axes[2].imshow(image, cmap='gray')
    if is_binary:
        overlay = np.ma.masked_where(pred_mask == 0, pred_mask)
        axes[2].imshow(overlay, cmap='Reds', alpha=0.5, vmin=0, vmax=1)
        axes[2].set_title('Overlay (Red = LV)')
    else:
        for label, cmap_name in [(1, 'Reds'), (2, 'Blues'), (3, 'Greens')]:
            overlay = np.ma.masked_where(pred_mask != label, pred_mask)
            axes[2].imshow(overlay, cmap=cmap_name, alpha=0.5, vmin=0, vmax=3)
        axes[2].set_title('Overlay (R=LV, B=Myo, G=LA)')
    axes[2].axis('off')

    im = axes[3].imshow(confidence, cmap='RdYlGn', vmin=0.5, vmax=1.0)
    axes[3].set_title(f'Confidence (avg: {confidence.mean():.2f})')
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046)

    if title:
        plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)

    plt.tight_layout()
    plt.show()

print("Visualization functions ready")

In [ ]:
# Cell [10] — Run on a Single Clip (all models)
# Your 15 probe clips: clip1.mp4 .. clip15.mp4  (plus clip3.png)
# Change the filename below to test any single clip.

test_clip = os.path.join(IMAGE_DIR, "clip1.mp4")  # <-- change clip number here

if os.path.exists(test_clip):
    # Show video info if it's an mp4
    if test_clip.lower().endswith(('.mp4', '.avi', '.mov')):
        get_video_info(test_clip)

    image, spacing = load_ultrasound_image(test_clip)
    print(f"\nLoaded: {test_clip}")
    print(f"  Frame shape: {image.shape}, Pixel spacing: {spacing} mm")

    # 1 — View classification
    view_idx, view_probs = predict_view_class(image)
    if view_idx is not None:
        print(f"\n[Classification] Predicted view index: {view_idx}  (confidence: {view_probs[view_idx]:.2f})")
    else:
        print("\n[Classification] Model not loaded — skipped")

    # 2 — Quality score
    quality = predict_quality_score(image)
    if quality is not None:
        print(f"[Quality]        Score: {quality:.3f}")
    else:
        print("[Quality]        Model not loaded — skipped")

    # 3 — Segmentation
    if "segmentation" in loaded_models:
        pred_mask, confidence = predict_segmentation(loaded_models["segmentation"], image)
        areas = calculate_areas(pred_mask, spacing)
        lv_length = calculate_lv_length_mm(pred_mask, spacing)
        print(f"\n[Segmentation]")
        print(f"  LV Cavity Area:    {areas['lv_area_mm2']:.1f} mm2")
        if 'myocardium_area_mm2' in areas:
            print(f"  Myocardium Area:   {areas['myocardium_area_mm2']:.1f} mm2")
            print(f"  Left Atrium Area:  {areas['la_area_mm2']:.1f} mm2")
        if lv_length:
            print(f"  LV Length:         {lv_length:.1f} mm")
        print(f"  Avg Confidence:    {confidence.mean():.2f}")
        visualize_result(image, pred_mask, confidence, title=Path(test_clip).name)
    else:
        print("\n[Segmentation]   Model not loaded — skipped")

    # 4 — Direct EF prediction
    ef_direct = predict_ef_direct(image)
    if ef_direct is not None:
        print(f"[EF Direct]      Predicted EF: {ef_direct:.1f}%")
    else:
        print("[EF Direct]      Model not loaded — skipped")
else:
    print(f"File not found: {test_clip}")
    print(f"Make sure IMAGE_DIR in Cell [3] points to your test_images/ folder")

In [ ]:
# Cell [11] — Full Pipeline: Process a Set of 15 Probe Images (all models)

def process_all_images(image_dir):
    """
    Run every loaded model on all ultrasound images in a directory.
    Returns a list of per-image result dicts.
    """
    supported_ext = ('.mp4', '.avi', '.mov', '.nii.gz', '.nii', '.png', '.jpg', '.jpeg', '.dcm', '.npy')

    image_files = []
    for f in sorted(os.listdir(image_dir)):
        if any(f.lower().endswith(ext) for ext in supported_ext):
            image_files.append(os.path.join(image_dir, f))

    if not image_files:
        print(f"No supported images found in {image_dir}")
        return []

    print(f"Found {len(image_files)} images in {image_dir}\n")

    results = []
    for i, img_path in enumerate(image_files, 1):
        fname = Path(img_path).name
        print(f"[{i}/{len(image_files)}] {fname}")

        try:
            image, spacing = load_ultrasound_image(img_path)
            entry = {
                'file': fname,
                'image': image,
                'spacing': spacing,
            }

            # Classification
            view_idx, view_probs = predict_view_class(image)
            entry['view_class'] = view_idx
            entry['view_probs'] = view_probs

            # Quality
            entry['quality_score'] = predict_quality_score(image)

            # Segmentation
            if "segmentation" in loaded_models:
                pred_mask, confidence = predict_segmentation(loaded_models["segmentation"], image)
                areas = calculate_areas(pred_mask, spacing)
                entry.update({
                    'pred_mask': pred_mask,
                    'confidence': confidence,
                    'areas': areas,
                    'lv_length_mm': calculate_lv_length_mm(pred_mask, spacing),
                    'avg_confidence': float(confidence.mean()),
                    'has_lv': bool((pred_mask == 1).any()),
                    'has_myocardium': bool((pred_mask == 2).any()),
                    'has_la': bool((pred_mask == 3).any()),
                })
                seg_info = f"LV={areas['lv_area_mm2']:.0f}mm2"
            else:
                seg_info = "seg N/A"

            # Direct EF
            entry['ef_direct'] = predict_ef_direct(image)

            qual_str = f"Q={entry['quality_score']:.2f}" if entry['quality_score'] is not None else "Q=N/A"
            view_str = f"View={view_idx}" if view_idx is not None else "View=N/A"
            ef_str = f"EF={entry['ef_direct']:.1f}%" if entry['ef_direct'] is not None else "EF=N/A"
            print(f"   {view_str} | {qual_str} | {seg_info} | {ef_str}")

            results.append(entry)

        except Exception as e:
            print(f"   Error: {e}")
            results.append({'file': fname, 'error': str(e)})

    return results

if os.path.isdir(IMAGE_DIR):
    all_results = process_all_images(IMAGE_DIR)
else:
    print(f"Directory not found: {IMAGE_DIR}")
    print(f"Create the folder and add your ultrasound images, or update IMAGE_DIR in Cell [3]")
    all_results = []

In [ ]:
# Cell [12] — Full Cardiac Report (all models)

def compute_full_cardiac_report(results, heart_rate_bpm=75, lvot_diameter_cm=2.0):
    """
    Print a comprehensive report using all available model outputs.
    """
    valid = [r for r in results if 'error' not in r]

    if not valid:
        print("No valid results to analyze")
        return None

    print("\n" + "=" * 80)
    print("  CARDIAC REPORT")
    print("=" * 80)

    # --- Per-image table ---
    has_class = any(r.get('view_class') is not None for r in valid)
    has_qual  = any(r.get('quality_score') is not None for r in valid)
    has_seg   = any('areas' in r for r in valid)
    has_ef    = any(r.get('ef_direct') is not None for r in valid)

    has_multiclass = any('myocardium_area_mm2' in r.get('areas', {}) for r in valid)

    header = f"{'#':<3} {'File':<30}"
    if has_class: header += f" {'View':>5}"
    if has_qual:  header += f" {'Qual':>6}"
    if has_seg:
        header += f" {'LV mm2':>8}"
        if has_multiclass:
            header += f" {'Myo mm2':>8} {'LA mm2':>8}"
        header += f" {'Conf':>6}"
    if has_ef:    header += f" {'EF%':>6}"
    print(f"\n{header}")
    print("-" * len(header))

    for i, r in enumerate(valid):
        row = f"{i:<3} {r['file']:<30}"
        if has_class:
            v = r.get('view_class')
            row += f" {v if v is not None else '-':>5}"
        if has_qual:
            q = r.get('quality_score')
            row += f" {q:>6.2f}" if q is not None else f" {'-':>6}"
        if has_seg and 'areas' in r:
            row += f" {r['areas']['lv_area_mm2']:>8.1f}"
            if has_multiclass:
                row += f" {r['areas'].get('myocardium_area_mm2', 0):>8.1f}"
                row += f" {r['areas'].get('la_area_mm2', 0):>8.1f}"
            row += f" {r.get('avg_confidence', 0):>6.2f}"
        elif has_seg:
            row += f" {'--':>8}"
            if has_multiclass:
                row += f" {'--':>8} {'--':>8}"
            row += f" {'--':>6}"
        if has_ef:
            ef = r.get('ef_direct')
            row += f" {ef:>6.1f}" if ef is not None else f" {'--':>6}"
        print(row)

    # --- Volume / EF / CO / VTI section ---
    print("\n" + "-" * 80)
    print("To compute volumes, EF, VTI, and cardiac output from segmentation,")
    print("identify which images correspond to 2CH/4CH at ED/ES phases.")
    print("Then uncomment and update the indices below:\n")
    print("# idx_2ch_ed = 0   # index in 'valid' for 2-chamber end-diastole")
    print("# idx_2ch_es = 1   # 2-chamber end-systole")
    print("# idx_4ch_ed = 2   # 4-chamber end-diastole")
    print("# idx_4ch_es = 3   # 4-chamber end-systole")
    print("#")
    print("# a2ch_ed = valid[idx_2ch_ed]['areas']['lv_area_mm2']")
    print("# a4ch_ed = valid[idx_4ch_ed]['areas']['lv_area_mm2']")
    print("# l_ed    = valid[idx_4ch_ed]['lv_length_mm']")
    print("# edv     = calculate_volume_biplane(a2ch_ed, a4ch_ed, l_ed)")
    print("#")
    print("# a2ch_es = valid[idx_2ch_es]['areas']['lv_area_mm2']")
    print("# a4ch_es = valid[idx_4ch_es]['areas']['lv_area_mm2']")
    print("# l_es    = valid[idx_4ch_es]['lv_length_mm']")
    print("# esv     = calculate_volume_biplane(a2ch_es, a4ch_es, l_es)")
    print("#")
    print("# sv  = edv - esv")
    print("# ef  = calculate_ef(edv, esv)")
    print("# co  = calculate_cardiac_output(sv, heart_rate_bpm)")
    print("# vti = calculate_vti(sv, lvot_diameter_cm)")

    return valid

if all_results:
    report = compute_full_cardiac_report(all_results, heart_rate_bpm=75, lvot_diameter_cm=2.0)

In [ ]:
# Cell [13] — Visualize All Results in a Grid

def visualize_all_results(results, cols=5):
    """Display all segmentation results in a grid."""
    valid = [r for r in results if 'error' not in r]
    
    if not valid:
        print("No valid results to display")
        return
    
    n = len(valid)
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1) if cols > 1 else np.array([[axes]])
    
    for i, r in enumerate(valid):
        row, col = i // cols, i % cols
        ax = axes[row, col]
        
        ax.imshow(r['image'], cmap='gray')
        if 'pred_mask' in r:
            mask = r['pred_mask']
            if mask.max() <= 1:
                overlay = np.ma.masked_where(mask == 0, mask)
                ax.imshow(overlay, cmap='Reds', alpha=0.5, vmin=0, vmax=1)
            else:
                for label, cmap_name in [(1, 'Reds'), (2, 'Blues'), (3, 'Greens')]:
                    overlay = np.ma.masked_where(mask != label, mask)
                    ax.imshow(overlay, cmap=cmap_name, alpha=0.5, vmin=0, vmax=3)

        conf_str = f"{r['avg_confidence']:.2f}" if 'avg_confidence' in r else "N/A"
        ax.set_title(f"{r['file']}\nConf: {conf_str}", fontsize=9)
        ax.axis('off')
    
    # Hide empty subplots
    for i in range(n, rows * cols):
        row, col = i // cols, i % cols
        axes[row, col].axis('off')
    
    plt.suptitle("Segmentation Results — All Images", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

if all_results:
    visualize_all_results(all_results)